# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nauman024/FlyRank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

Unit of Analysis (Grain): One row represents one pseudonymized content item (page) for a given client aggregated over a target timeframe (or daily performance per page in fact_content_daily_performance).

Tables Used: Primary fact table fact_content_daily_performance joined with dim_content and dim_clients.

Time Window: Mid-panel month month = '2026-03' for feature engineering / historical observation, keeping 2026-06 strictly sealed.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

Target / Proxy Label: Binary underperformance risk flag is_underperforming (e.g., 1 if traffic/impressions drop $>20\%$ relative to previous period).

Deliberate Exclusion: Pages with impressions == 0 over the entire observation period or pages with is_active IS FALSE are excluded to prevent baseline dilution from dead/archived pages.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [14]:
import os
import duckdb
from google.colab import userdata

# 1. Connect to DuckDB & Register Hugging Face Secret
con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

# -------------------------------------------------------------------
# QUERY 1: Prove the Grain (One row per page/client on a mid-panel month)
# -------------------------------------------------------------------
query_1 = f"""
SELECT
    client_hash_id,
    content_hash_id,
    COUNT(*) as daily_records
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY client_hash_id, content_hash_id
LIMIT 5;
"""
print("--- Query 1: Grain Check (Month 2026-03) ---")
df_q1 = con.sql(query_1).df()
display(df_q1)

# -------------------------------------------------------------------
# QUERY 2: Row Count & Date Span for Mid-Panel Month (2026-03)
# -------------------------------------------------------------------
query_2 = f"""
SELECT
    COUNT(*) as total_rows,
    MIN(report_date) as min_date,
    MAX(report_date) as max_date,
    COUNT(DISTINCT content_hash_id) as unique_pages
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet');
"""
print("\n--- Query 2: Slice Row Count & Date Span ---")
df_q2 = con.sql(query_2).df()
display(df_q2)

# -------------------------------------------------------------------
# QUERY 3: Availability Check (IS FALSE / Availability filtering)
# -------------------------------------------------------------------
query_3 = f"""
SELECT
    COUNT(*) as active_surviving_rows
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') f
JOIN read_parquet('{rel}/dim_content.parquet') c ON f.content_hash_id = c.content_hash_id
WHERE c.is_deleted IS FALSE;
"""
print("\n--- Query 3: Availability Filter (is_deleted IS FALSE) ---")
df_q3 = con.sql(query_3).df()
display(df_q3)

--- Query 1: Grain Check (Month 2026-03) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,daily_records
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,31
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,31
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,31
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,31
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,31



--- Query 2: Slice Row Count & Date Span ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,min_date,max_date,unique_pages
0,9841378,2026-03-01,2026-03-31,331437



--- Query 3: Availability Filter (is_deleted IS FALSE) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,active_surviving_rows
0,9640288


In [15]:
# -------------------------------------------------------------------
# BUILD 5-FEATURE FRAME (Knowable at Decision Moment)
# -------------------------------------------------------------------
# Feature 1: avg_position_30d  -> Knowable: Aggregated past Google Search Console position.
# Feature 2: total_impressions -> Knowable: Logged search exposure in March 2026.
# Feature 3: total_clicks      -> Knowable: Historical user clicks in observation period.
# Feature 4: historical_ctr    -> Knowable: Derived from clicks / impressions ratio.
# Feature 5: is_deleted_flag   -> Knowable: Boolean metadata status from dim_content.

feature_query = f"""
SELECT
    f.content_hash_id,
    AVG(f.gsc_avg_position) as avg_position_30d,
    SUM(f.gsc_impressions) as total_impressions,
    SUM(f.gsc_clicks) as total_clicks,
    (SUM(f.gsc_clicks) / NULLIF(SUM(f.gsc_impressions), 0)) as historical_ctr,
    MAX(CASE WHEN c.is_deleted IS TRUE THEN 1 ELSE 0 END) as is_deleted_flag,
    -- PROXY TARGET
    CASE WHEN SUM(f.gsc_clicks) < 10 THEN 1 ELSE 0 END as target_underperforming
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') f
JOIN read_parquet('{rel}/dim_content.parquet') c ON f.content_hash_id = c.content_hash_id
WHERE c.is_deleted IS FALSE
GROUP BY f.content_hash_id
LIMIT 5000;
"""

feature_df = con.sql(feature_query).df().fillna(0)
print("--- 5-Feature Frame Created ---")
display(feature_df.head())

# -------------------------------------------------------------------
# THE TRAP: Adding ONE Label-Derived Column (Target Leakage)
# -------------------------------------------------------------------
feature_df['LEAKED_future_clicks'] = feature_df['target_underperforming'] * 0.1

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X_leaked = feature_df[['avg_position_30d', 'total_impressions', 'historical_ctr', 'LEAKED_future_clicks']]
y = feature_df['target_underperforming']

model_leaked = DecisionTreeClassifier()
model_leaked.fit(X_leaked, y)
leaked_score = accuracy_score(y, model_leaked.predict(X_leaked))

print(f"\n[LEAKAGE EXPERIMENT] Accuracy WITH Leaked Column: {leaked_score:.4f} (Artificially Perfect!)")

# -------------------------------------------------------------------
# CLEANUP: Deleting Leaked Column to Keep Honest Score
# -------------------------------------------------------------------
del feature_df['LEAKED_future_clicks']

X_honest = feature_df[['avg_position_30d', 'total_impressions', 'historical_ctr', 'is_deleted_flag']]
model_honest = DecisionTreeClassifier(max_depth=3)
model_honest.fit(X_honest, y)
honest_score = accuracy_score(y, model_honest.predict(X_honest))

print(f"[HONEST MODEL] Accuracy AFTER Removing Leaked Column: {honest_score:.4f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- 5-Feature Frame Created ---


,content_hash_id,avg_position_30d,total_impressions,total_clicks,historical_ctr,is_deleted_flag,target_underperforming
0,content_39d7361b4945d504,4.074107,77.0,0.0,0.000000,0,1
1,content_cec711b02f3bbde6,4.428747,602.0,4.0,0.006645,0,1
2,content_275b6f7f733016d4,4.866123,810.0,1.0,0.001235,0,1
3,content_ceaec531566ffcfc,8.978086,82.0,0.0,0.000000,0,1
4,content_755d951187fcd70a,1.854929,1858.0,6.0,0.003229,0,1



[LEAKAGE EXPERIMENT] Accuracy WITH Leaked Column: 1.0000 (Artificially Perfect!)
[HONEST MODEL] Accuracy AFTER Removing Leaked Column: 0.9902


## 4. Data limits

Limitation: Unbalanced client panel depth: Newer clients in dim_clients have shorter historical performance logs, leading to higher variance in rolling baseline features compared to established clients.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.